In [ ]:
# Libraries
library(tidyverse)

In [2]:
# List files
danish <- list.files("12-ExtractTaxaReads/Danish", pattern = ".tsv", recursive = T, full.names = T)
exotic <- list.files("12-ExtractTaxaReads/Exotic", pattern = ".tsv", recursive = T, full.names = T)

In [3]:
# Read data: danish (~3 min)
danish <- lapply(danish, function(x) read.table(x, header=T) %>%
         mutate(category = str_split_i(x, "/", 2),
                sample = str_split_i(x, "/", 3),
                phylum = str_split_i(str_replace(str_split_i(x, "/", 4), ".tsv", ""), "_", 2))
)

In [4]:
# Read data: exotic
exotic <- lapply(exotic, function(x) read.table(x, header=T) %>%
                   mutate(category = str_split_i(x, "/", 2),
                          sample = str_split_i(x, "/", 3),
                          phylum = str_split_i(str_replace(str_split_i(x, "/", 4), ".tsv", ""), "_", 2))
)

In [ ]:
# Reduce data (~2 min)
danish <- Reduce(function(...) rbind(...), danish)
exotic <- Reduce(function(...) rbind(...), exotic)

In [7]:
# Merge data
df <- rbind(danish, exotic)

In [11]:
# Rename samples
df <- df %>%
  mutate(sample = recode(sample, 
                         "GP_1" = "EN0.2A", "GP_2" = "EN0.2B", "GP_3" = "EN0.2C",
                         "A0_2" = "OP0.2A", "B0_2" = "OP0.2B", "C0_2" = "OP0.2C",
                         "A1_2" = "OP1.2A", "B1_2" = "OP1.2B", "C1_2" = "OP1.2C",
                         "A5_0" = "OP5.0A", "B5_0" = "OP5.0B", "C5_0" = "OP5.0C",
                         "A8_0" = "OP8.0A", "B8_0" = "OP8.0B", "C8_0" = "OP8.0C"))

In [28]:
# Compute correlation (More Danish == More Exotic?)
df_cor <- df %>% group_by(category, sample, phylum) %>% summarize(no_of_reads = n())
df_cor <- df_cor %>% pivot_wider(id_cols = c(sample,phylum), names_from = category, values_from = no_of_reads)
df_cor <- df_cor %>% mutate(filter = recode(sample, 
                         "EN0.2A" = "EN0.2", "EN0.2B" = "EN0.2", "EN0.2C" = "EN0.2",
                         "OP0.2A" = "OP0.2", "OP0.2B" = "OP0.2", "OP0.2C" = "OP0.2",
                         "OP1.2A" = "OP1.2", "OP1.2B" = "OP1.2", "OP1.2C" = "OP1.2",
                         "OP5.0A" = "OP5.0", "OP5.0B" = "OP5.0", "OP5.0C" = "OP5.0",
                         "OP8.0A" = "OP8.0", "OP8.0B" = "OP8.0", "OP8.0C" = "OP8.0"))
head(df_cor)

`summarise()` has grouped output by 'category', 'sample'. You can override using the `.groups` argument.


sample,phylum,Danish,Exotic,filter
<chr>,<chr>,<int>,<int>,<chr>
EN0.2A,Annelida,218,2,EN0.2
EN0.2A,Arthropoda,79,83,EN0.2
EN0.2A,Chordata,1012090,7749,EN0.2
EN0.2A,Mollusca,3688,621,EN0.2
EN0.2B,Annelida,119,1,EN0.2
EN0.2B,Arthropoda,168,80,EN0.2


In [51]:
# Correlation per phylum
df_cor %>%
  group_by(phylum) %>%
  summarise(correlation = cor(Danish, Exotic, method = "pearson", use = "complete.obs")*100)

phylum,correlation
<chr>,<dbl>
Annelida,98.81431
Arthropoda,-19.65818
Chordata,71.29148
Mollusca,95.19238


In [52]:
# Correlation per filter
df_cor %>%
  group_by(filter) %>%
  summarise(correlation = cor(Danish, Exotic, method = "pearson", use = "complete.obs")*100)

filter,correlation
<chr>,<dbl>
EN0.2,99.67999
OP0.2,98.50618
OP1.2,83.12926
OP5.0,92.92692
OP8.0,98.60601


In [104]:
# Plot correlation pattern
p <- df_cor %>% ggplot(aes(x=log(Exotic), y=log(Danish), fill=filter, shape=phylum, color=filter)) +
geom_point(size=2) +
labs(color="Filter type", shape="Phylum", x="No .of Exotic reads (Log)", y="No. of Danish reads (Log)") +
guides(fill="none") +
scale_shape_manual(values = c(21,25,24,22)) +
scale_color_manual(values = c('#f8766d','#b79f00','#00bf7d','#00b0f6','#e76bf3')) +
scale_fill_manual(values = c('#f8766d','#b79f00','#00bf7d','#00b0f6','#e76bf3')) +
theme_bw(base_size = 12)

png(file="12-ExtractTaxaReads/cor_reads.danish_vs_exotic.png", width = 1500, height = 1200, res=300)
p
dev.off()

Warning message:
“Removed 1 rows containing missing values (`geom_point()`).”


png 
  2

In [8]:
# Summarize data
nreads <- df %>% group_by(category, sample, phylum) %>% summarize(count = n()) %>%
mutate(group = recode(sample, 
                         "EN0.2A" = "EN0.2", "EN0.2B" = "EN0.2", "EN0.2C" = "EN0.2",
                         "OP0.2A" = "OP0.2", "OP0.2B" = "OP0.2", "OP0.2C" = "OP0.2",
                         "OP1.2A" = "OP1.2", "OP1.2B" = "OP1.2", "OP1.2C" = "OP1.2",
                         "OP5.0A" = "OP5.0", "OP5.0B" = "OP5.0", "OP5.0C" = "OP5.0",
                         "OP8.0A" = "OP8.0", "OP8.0B" = "OP8.0", "OP8.0C" = "OP8.0"))
nreads$group <- factor(nreads$group, levels = c('EN0.2', 'OP0.2', 'OP1.2', 'OP5.0', 'OP8.0'))
head(nreads)

`summarise()` has grouped output by 'category', 'sample'. You can override using the `.groups` argument.


category,sample,phylum,count,group
<chr>,<chr>,<chr>,<int>,<fct>
Danish,EN0.2A,Annelida,218,EN0.2
Danish,EN0.2A,Arthropoda,79,EN0.2
Danish,EN0.2A,Chordata,1012090,EN0.2
Danish,EN0.2A,Mollusca,3688,EN0.2
Danish,EN0.2B,Annelida,119,EN0.2
Danish,EN0.2B,Arthropoda,168,EN0.2


In [22]:
# Plot read counts
p <- nreads %>% ggplot() + geom_boxplot(aes(x=group, y=log10(count), fill=group)) +
facet_grid(phylum~category, scales = "free") +
ylim(0,NA) +
labs(x="Phylum", y="No. of reads (Log10)", fill="Filter type") +
theme_bw() %+replace% theme(axis.text.x = element_text(angle = 90))

png(file="12-ExtractTaxaReads/num_reads.png", width = 1500, height = 1200, res=300)
p
dev.off()

png 
  2

In [ ]:
# Plot e-value (zero-handling) distributions
p <- df %>% ggplot() + geom_histogram(aes(x=-log10(evalue + 1e-300), fill=category), alpha=0.75, bins = 30) +
facet_grid(phylum~sample, scales = "free") +
scale_fill_manual(values = c("darkred","gray25")) +
labs(x="E-value (-Log10)", y="No. of reads", fill="Category") +
theme_bw(base_size = 12) %+replace% theme(legend.position = "top")

png(file="12-ExtractTaxaReads/evalue_per_sample.png", width = 4000, height = 1800, res=300)
p
dev.off()

png 
  2

In [ ]:
# Plot pident distributions
p <- df %>% ggplot() + geom_histogram(aes(x=pident, fill=category), alpha=0.75, bins = 30) +
facet_grid(phylum~sample, scales = "free") +
scale_fill_manual(values = c("darkred","gray25")) +
labs(x="Percentage of identity", y="No. of reads", fill="Category") +
theme_bw(base_size = 12) %+replace% theme(legend.position = "top",
										  axis.text.x = element_text(angle = 45, size = 8, vjust=1, hjust=1))

png(file="12-ExtractTaxaReads/pident_per_sample.png", width = 4000, height = 1800, res=300)
p
dev.off()

png 
  2

In [24]:
# plot length vs mismatch (~7 min)
p <- df %>% ggplot() + geom_point(aes(y=as.numeric(length), x=as.numeric(mismatch), color=category), alpha=0.25) +
facet_grid(phylum~sample, scales = "free") +
scale_color_manual(values = c("darkred","gray25")) +
labs(x="No. of mistmatches", y="Alignment length", fill="Category") +
theme_bw(base_size = 12) %+replace% theme(legend.position = "top")

png(file="12-ExtractTaxaReads/aln_vs_mismatch_per_sample.png", width = 4000, height = 1800, res=300)
p
dev.off()

png 
  2